# Representative Policies vs Random: Hierarchical Analysis

This notebook analyzes the **representative family** of policies against random from Plan B summaries.

It reports:
- Dataset-level **hierarchical bootstrap deltas** (subset -> task -> dataset).
- **Hierarchical ECDFs** of subset-level deltas (bootstrap with subset resampling within task, then task-averaging).

Convention used here: deltas are sign-normalized so **positive is better** for every metric.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys


def resolve_repo_root(start: Path | None = None) -> Path:
    probe = (start or Path.cwd()).resolve()
    for cand in [probe, *probe.parents]:
        if (cand / 'experiments' / 'scripts').exists() and (cand / 'experiments' / 'analysis').exists():
            return cand
    raise FileNotFoundError(
        f"Could not resolve repo root from cwd={Path.cwd()}. "
        "Expected to find experiments/scripts and experiments/analysis in a parent directory."
    )


REPO_ROOT = resolve_repo_root()
for path in [REPO_ROOT, REPO_ROOT / 'UniverSeg', REPO_ROOT / 'MultiverSeg']:
    if str(path) not in sys.path:
        sys.path.append(str(path))

from experiments.analysis.planb_utils import load_planb_summaries
from experiments.analysis.hierarchical_ci import hierarchical_bootstrap_grouped_deltas

FIG_DIR = REPO_ROOT / 'figures' / 'representative_vs_random_hierarchical'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print('Resolved REPO_ROOT:', REPO_ROOT)
pd.set_option('display.max_columns', 200)


Resolved REPO_ROOT: /data/ddmg/mvseg-ordering


In [2]:
# ---- Config ----
procedure = 'random_v_repr_final'
ablation = 'pretrained_baseline5p'
dataset = None                  # e.g. 'STARE' or None for all families
mega_slicing = None             # 'midslice' / 'maxslice' / None

random_policy = 'random'
representative_policies = None  # None => auto-detect policy_name containing 'representative'

# metric -> direction (higher/lower better)
metric_directions = {
    'final_dice': 'higher',
    'iterations_used': 'lower',
}

n_boot = 2000
seed = 0
alpha = 0.05

save_tables = True
out_prefix = f"{procedure}__{ablation}__{dataset or 'all'}"


In [ ]:
def build_subset_delta_table(
    full_df: pd.DataFrame,
    *,
    metric: str,
    direction: str,
    random_policy: str,
    representative_policies: list[str],
) -> pd.DataFrame:
    keys = ['family', 'task_id', 'subset_index', 'policy_name', 'permutation_index']
    required = set(keys + [metric])
    missing = required - set(full_df.columns)
    if missing:
        raise ValueError(f"Missing required columns for metric={metric}: {sorted(missing)}")

    per_perm = full_df.groupby(keys, as_index=False)[metric].mean()
    subset_means = (
        per_perm.groupby(['family', 'task_id', 'subset_index', 'policy_name'], as_index=False)[metric]
        .mean()
        .rename(columns={metric: 'subset_metric'})
    )

    rand = subset_means[subset_means['policy_name'] == random_policy].copy()
    rand = rand.rename(columns={'subset_metric': 'random_subset_metric'})
    rand = rand[['family', 'task_id', 'subset_index', 'random_subset_metric']]

    rows = []
    for policy in representative_policies:
        rep = subset_means[subset_means['policy_name'] == policy].copy()
        if rep.empty:
            continue
        rep = rep.rename(columns={'subset_metric': 'rep_subset_metric'})
        rep = rep[['family', 'task_id', 'subset_index', 'rep_subset_metric']]

        merged = rep.merge(rand, on=['family', 'task_id', 'subset_index'], how='inner')
        if merged.empty:
            continue

        if direction == 'higher':
            delta = merged['rep_subset_metric'] - merged['random_subset_metric']
        elif direction == 'lower':
            delta = merged['random_subset_metric'] - merged['rep_subset_metric']
        else:
            raise ValueError(f"Unknown direction: {direction}")

        out = merged.copy()
        out['metric'] = metric
        out['direction'] = direction
        out['policy_name'] = policy
        out['delta_vs_random'] = delta
        rows.append(out)

    if not rows:
        return pd.DataFrame()

    out_df = pd.concat(rows, ignore_index=True)
    out_df['task_name'] = out_df['task_id'].astype(str).str.split('/', n=1).str[-1]
    return out_df


full_df = load_planb_summaries(
    repo_root=REPO_ROOT,
    procedure=procedure,
    ablation=ablation,
    dataset=dataset,
    mega_slicing=mega_slicing,
    filename='subset_support_images_summary.csv',
    allow_root_fallback=True,
)

all_policies = sorted(full_df['policy_name'].dropna().astype(str).unique().tolist())
if representative_policies is None:
    representative_policies = [p for p in all_policies if 'representative' in p.lower()]

print('Policies found:', all_policies)
print('Representative policies used:', representative_policies)

missing_metrics = [m for m in metric_directions if m not in full_df.columns]
if missing_metrics:
    raise ValueError(f"Metrics not found in summaries: {missing_metrics}")

delta_frames = []
for metric, direction in metric_directions.items():
    frame = build_subset_delta_table(
        full_df,
        metric=metric,
        direction=direction,
        random_policy=random_policy,
        representative_policies=representative_policies,
    )
    if not frame.empty:
        delta_frames.append(frame)

if not delta_frames:
    raise ValueError('No paired representative-vs-random subset deltas were produced.')

delta_df = pd.concat(delta_frames, ignore_index=True)
print('delta_df shape:', delta_df.shape)
delta_df.head(10)


In [ ]:
bootstrap_result = hierarchical_bootstrap_grouped_deltas(
    delta_df,
    value_col='delta_vs_random',
    group_cols=['metric', 'policy_name'],
    dataset_col='family',
    task_col='task_id',
    n_boot=n_boot,
    seed=seed,
    alpha=alpha,
)

dataset_summary = bootstrap_result['dataset_summary'].copy()
global_summary = bootstrap_result['global_summary'].copy()

display(dataset_summary)
display(global_summary)


In [ ]:
if save_tables:
    delta_out = FIG_DIR / f'{out_prefix}__subset_deltas.csv'
    summary_out = FIG_DIR / f'{out_prefix}__dataset_bootstrap_summary.csv'
    delta_df.to_csv(delta_out, index=False)
    dataset_summary.to_csv(summary_out, index=False)
    print('Saved', delta_out)
    print('Saved', summary_out)


In [ ]:
# Dataset-grid hierarchical bootstrap delta bars (one panel per dataset family)
for metric in sorted(dataset_summary['metric'].unique().tolist()):
    metric_df = dataset_summary[dataset_summary['metric'] == metric].copy()
    if metric_df.empty:
        continue

    families = sorted(metric_df['family'].dropna().astype(str).unique().tolist())
    if not families:
        continue

    n_fam = len(families)
    n_cols = min(4, n_fam)
    n_rows = int(np.ceil(n_fam / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(4.6 * n_cols, 3.8 * n_rows),
        squeeze=False,
        sharey=True,
    )
    axes_flat = axes.flatten()

    for ax_idx, family in enumerate(families):
        ax = axes_flat[ax_idx]
        fam_df = metric_df[metric_df['family'] == family].copy()
        fam_df = fam_df.sort_values('policy_name').reset_index(drop=True)
        if fam_df.empty:
            ax.set_visible(False)
            continue

        labels = fam_df['policy_name'].astype(str).tolist()
        x = np.arange(len(labels))
        means = fam_df['mean'].to_numpy(dtype=float)
        lo = fam_df['ci_lo'].to_numpy(dtype=float)
        hi = fam_df['ci_hi'].to_numpy(dtype=float)
        yerr = [means - lo, hi - means]

        ax.bar(x, means, yerr=yerr, capsize=4)
        ax.axhline(0.0, color='black', linewidth=0.8)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha='right')
        ax.set_title(family)
        ax.grid(axis='y', alpha=0.25)

    for ax in axes_flat[n_fam:]:
        ax.set_visible(False)

    fig.suptitle(f'Dataset-grid hierarchical deltas vs random: {metric}', y=0.995)
    for row_axes in axes:
        row_axes[0].set_ylabel('Delta vs random (positive = better)')
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    out_path = FIG_DIR / f'{out_prefix}__dataset_grid_bootstrap_delta__{metric}.png'
    fig.savefig(out_path, dpi=220)
    plt.show()
    plt.close(fig)
    print('Saved', out_path)


In [ ]:
def hierarchical_bootstrap_ecdf(
    subset_scores_by_task: dict[str, np.ndarray],
    x_grid: np.ndarray,
    *,
    n_boot: int,
    seed: int,
    alpha: float,
) -> dict[str, object]:
    tasks = sorted(subset_scores_by_task.keys())
    if not tasks:
        raise ValueError('No tasks provided for hierarchical ECDF bootstrap.')

    boot_curves = []
    for i in range(int(n_boot)):
        rng = np.random.default_rng(int(seed) + i)
        task_curves = []
        for task in tasks:
            vals = np.asarray(subset_scores_by_task[task], dtype=float)
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                continue
            idx = rng.integers(0, vals.size, size=vals.size)
            sample = vals[idx]
            # ECDF F(x)=P(Delta<=x)
            cdf = np.mean(sample[:, None] <= x_grid[None, :], axis=0)
            task_curves.append(cdf)
        if task_curves:
            boot_curves.append(np.mean(task_curves, axis=0))

    if not boot_curves:
        raise ValueError('Failed to build ECDF bootstrap curves.')

    boot = np.vstack(boot_curves)
    return {
        'x': x_grid,
        'mean': boot.mean(axis=0),
        'ci_lo': np.quantile(boot, alpha / 2.0, axis=0),
        'ci_hi': np.quantile(boot, 1.0 - alpha / 2.0, axis=0),
        'boot_curves': boot,
        'n_boot': boot.shape[0],
    }


# Hierarchical ECDF curves per (metric, family), one curve per representative policy.
ecdf_rows = []
for metric in sorted(delta_df['metric'].unique().tolist()):
    metric_df = delta_df[delta_df['metric'] == metric]
    for family in sorted(metric_df['family'].unique().tolist()):
        fam_df = metric_df[metric_df['family'] == family]
        if fam_df.empty:
            continue

        pooled = fam_df['delta_vs_random'].to_numpy(dtype=float)
        pooled = pooled[np.isfinite(pooled)]
        if pooled.size == 0:
            continue

        lo, hi = np.quantile(pooled, [0.01, 0.99])
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            lo = float(np.nanmin(pooled))
            hi = float(np.nanmax(pooled))
            if lo == hi:
                lo -= 1e-6
                hi += 1e-6
        x_grid = np.linspace(lo, hi, 250)

        fig, ax = plt.subplots(figsize=(7.2, 4.8))
        for policy in sorted(fam_df['policy_name'].unique().tolist()):
            pol_df = fam_df[fam_df['policy_name'] == policy]
            subset_scores_by_task = {
                str(task_id): grp['delta_vs_random'].to_numpy(dtype=float)
                for task_id, grp in pol_df.groupby('task_id', sort=True)
            }
            subset_scores_by_task = {
                t: vals[np.isfinite(vals)]
                for t, vals in subset_scores_by_task.items()
                if np.isfinite(vals).any()
            }
            if not subset_scores_by_task:
                continue

            ecdf = hierarchical_bootstrap_ecdf(
                subset_scores_by_task,
                x_grid,
                n_boot=n_boot,
                seed=seed,
                alpha=alpha,
            )

            ax.plot(ecdf['x'], ecdf['mean'], linewidth=2.0, label=f'{policy}')
            ax.fill_between(ecdf['x'], ecdf['ci_lo'], ecdf['ci_hi'], alpha=0.15)

            for x, m, lo_y, hi_y in zip(ecdf['x'], ecdf['mean'], ecdf['ci_lo'], ecdf['ci_hi']):
                ecdf_rows.append({
                    'metric': metric,
                    'family': family,
                    'policy_name': policy,
                    'delta_threshold': float(x),
                    'ecdf_mean': float(m),
                    'ecdf_ci_lo': float(lo_y),
                    'ecdf_ci_hi': float(hi_y),
                })

        ax.axvline(0.0, color='black', linestyle='--', linewidth=1.0, label='Random baseline (delta=0)')
        ax.set_xlabel('Delta vs random (positive = better)')
        ax.set_ylabel('Hierarchical ECDF  F(Delta ≤ x)')
        ax.set_title(f'Hierarchical ECDF of subset deltas: {family} / {metric}')
        ax.grid(alpha=0.3)
        ax.legend(loc='best')
        fig.tight_layout()

        out_path = FIG_DIR / f'{out_prefix}__hier_ecdf__{family}__{metric}.png'
        fig.savefig(out_path, dpi=220)
        plt.show()
        plt.close(fig)
        print('Saved', out_path)

ecdf_df = pd.DataFrame(ecdf_rows)
if save_tables and not ecdf_df.empty:
    ecdf_out = FIG_DIR / f'{out_prefix}__hier_ecdf_curves.csv'
    ecdf_df.to_csv(ecdf_out, index=False)
    print('Saved', ecdf_out)

print('ecdf_df shape:', ecdf_df.shape)
ecdf_df.head(10)
